<div style='text-align: center; background-color: #1a1a2e; color: white; padding: 30px; border-radius: 12px; font-family: Arial, sans-serif;'>

<h1 style='color: #e94560; margin-bottom: 5px;'>🌐 Internet de las Cosas (IoT)</h1>
<h2 style='color: #ffffff; margin-top: 0;'>Actividad 3 — Sistema de Transmisión de Datos</h2>

<hr style='border-color: #e94560; margin: 20px 0;'/>

<table style='margin: 0 auto; font-size: 16px; color: #cccccc;'>
<tr><td style='padding: 6px 20px; text-align: right; color: #e94560;'><b>Estudiante:</b></td><td style='padding: 6px 20px; text-align: left;'>Ana María García Arias</td></tr>
<tr><td style='padding: 6px 20px; text-align: right; color: #e94560;'><b>Programa:</b></td><td style='padding: 6px 20px; text-align: left;'>Maestría en Inteligencia Artificial</td></tr>
<tr><td style='padding: 6px 20px; text-align: right; color: #e94560;'><b>Asignatura:</b></td><td style='padding: 6px 20px; text-align: left;'>Internet de las Cosas</td></tr>
<tr><td style='padding: 6px 20px; text-align: right; color: #e94560;'><b>Docente:</b></td><td style='padding: 6px 20px; text-align: left;'>Cristian Duney Bermúdez Quintero</td></tr>
<tr><td style='padding: 6px 20px; text-align: right; color: #e94560;'><b>Fecha:</b></td><td style='padding: 6px 20px; text-align: left;'>Mayo 2026</td></tr>
</table>

</div>

---

## 🎬 Video explicativo

El siguiente video presenta el proceso de implementación, la configuración del sistema y los resultados obtenidos:

[![Ver video explicativo](https://img.youtube.com/vi/0L9Anxb1Pzk/0.jpg)](https://youtu.be/0L9Anxb1Pzk)

**▶ [https://youtu.be/0L9Anxb1Pzk](https://youtu.be/0L9Anxb1Pzk)**

---

## 1. Objetivo

Implementar un sistema de transmisión de datos IoT donde un nodo sensor captura información de temperatura y humedad relativa, la almacena localmente en una base de datos y la transmite en tiempo real a una plataforma de monitoreo en la nube mediante el protocolo **MQTT**.

---

## 2. Arquitectura del sistema

```
┌─────────────────────────────────────────────────────────────────────┐
│                        NODO SENSOR VIRTUAL                          │
│                                                                     │
│  CounterFit (sensor DHT11 virtual)                                  │
│    ├─ Humidity  Pin 5  → Random [40 – 90 %]                         │
│    └─ Temperature Pin 6 → Random [18 – 35 °C]                       │
│              │                                                      │
│              ▼                                                      │
│       nodo_mqtt.py  (Python 3.11 — PC actúa como nodo IoT)          │
│              │                                                      │
│    ┌─────────┴──────────┐                                           │
│    ▼                    ▼                                           │
│  SQLite              paho-mqtt                                      │
│  (demo.db)           (protocolo MQTT)                               │
│  Backup local            │                                          │
└──────────────────────────│──────────────────────────────────────────┘
                           │  Wi-Fi / Internet
                           ▼
               Broker: io.adafruit.com:1883
                           │
               ┌───────────┴────────────┐
               ▼                        ▼
    Feed: temperatura           Feed: humedad
               └───────────┬────────────┘
                           ▼
              Dashboard Adafruit IO
           (visualización en tiempo real)
```

El PC cumple el mismo rol que cumpliría un microcontrolador **ESP32** o **ESP8266** en una implementación con hardware físico: leer el sensor, persistir los datos y publicarlos vía MQTT. El protocolo, el broker y la integración con la plataforma son idénticos.

---

## 3. Protocolo de comunicación: MQTT

**MQTT** (*Message Queuing Telemetry Transport*) es un protocolo de mensajería ligero diseñado para entornos IoT. Opera bajo el modelo **publicar / suscribir** (*publish / subscribe*), que desacopla al emisor del receptor a través de un intermediario llamado **broker**.

| Concepto | Descripción |
|---|---|
| **Broker** | Servidor intermediario que enruta los mensajes. En este proyecto: `io.adafruit.com:1883` |
| **Topic (feed)** | Canal lógico de datos. En Adafruit IO: `NANY1993/feeds/temperatura` y `NANY1993/feeds/humedad` |
| **Publisher** | El nodo sensor (`nodo_mqtt.py`) que publica los valores |
| **Subscriber** | Adafruit IO, que recibe y grafica los valores en el dashboard |
| **QoS 0** | *At most once* — envío sin confirmación, suficiente para telemetría continua |
| **Payload** | Valor numérico en texto (ej. `"24.53"`) |

### ¿Por qué MQTT y no HTTP o CoAP?

| Protocolo | Peso | Modelo | Uso ideal |
|---|---|---|---|
| **MQTT** ✅ | Muy ligero | Pub/Sub — conexión persistente | Telemetría IoT continua |
| HTTP | Medio | Request/Response | APIs REST |
| CoAP | Ligero | Request/Response sobre UDP | Redes muy restringidas |

MQTT mantiene una sola conexión TCP abierta durante toda la sesión, lo que lo hace ideal para sensores que envían datos de forma periódica.

---

## 4. Herramientas y tecnologías utilizadas

| Herramienta | Versión | Rol en el proyecto |
|---|---|---|
| **Python** | 3.11 | Lenguaje principal del nodo sensor |
| **CounterFit** | latest | Simulador de sensores IoT (DHT11 virtual) |
| **counterfit-shims-seeed-python-dht** | latest | Shim que permite leer el DHT11 virtual como si fuera físico |
| **paho-mqtt** | ≥1.6 | Cliente MQTT para publicar datos a Adafruit IO |
| **SQLite** | integrado en Python | Base de datos local para persistencia |
| **python-dotenv** | ≥1.0 | Gestión segura de credenciales desde archivo `.env` |
| **tqdm** | ≥4.66 | Barra de progreso en consola |
| **Adafruit IO** | cloud | Plataforma de monitoreo con dashboard en tiempo real |
| **Red Wi-Fi** | TCP/IP | Tipo de red utilizada para la transmisión |


---

## 5. Configuración del sistema

### Sensor virtual (CounterFit)

| Sensor | Tipo | Unidades | Pin | Modo | Min | Max |
|---|---|---|---|---|---|---|
| Humedad | Humidity | Percentage | 5 | Random | 40 | 90 |
| Temperatura | Temperature | Celsius | 6 | Random | 18 | 35 |

> **Nota:** CounterFit requiere que el campo *Value* tenga un número antes de hacer clic en *Set*, incluso cuando se usa el modo Random. Si el campo queda vacío, el servidor falla silenciosamente y el modo aleatorio no se activa.

### Plataforma de monitoreo (Adafruit IO)

- **Broker MQTT:** `io.adafruit.com`, puerto `1883`
- **Feeds configurados:** `temperatura` y `humedad`
- **Dashboard:** bloques *Line Chart* y *Gauge* para cada variable
- **Credenciales:** gestionadas con `.env` (excluido de GitHub por seguridad)

### Comando de ejecución

```powershell
python nodo_mqtt.py --interval-sec 10 --samples 1080 --db demo.db
```

Equivalente a 3 horas de muestreo continuo (1080 lecturas × 10 segundos).

---

## 6. Resultados obtenidos

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Cargar datos desde SQLite
con = sqlite3.connect("demo.db")
df = pd.read_sql_query(
    "SELECT id, ts, temperatura_c, humedad_pct FROM lecturas ORDER BY id",
    con
)
con.close()

df["tiempo"] = pd.to_datetime(df["ts"])

print(f"Total de registros en la base de datos: {len(df)}")
print(f"Primera lectura : {df['tiempo'].min()}")
print(f"Última lectura  : {df['tiempo'].max()}")
df.tail(10)

In [ ]:
# Estadísticos descriptivos
stats = df[["temperatura_c", "humedad_pct"]].agg(["count", "min", "max", "mean", "std"]).round(2)
stats.index = ["n lecturas", "Mínimo", "Máximo", "Media", "Desv. estándar"]
stats.columns = ["Temperatura (°C)", "Humedad relativa (%)"]
print("Tabla de estadísticos descriptivos")
stats

In [ ]:
# Series temporales
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
fig.suptitle("Actividad 3 — Nodo Sensor IoT: Series temporales", fontsize=14, fontweight="bold")

axes[0].plot(df["tiempo"], df["temperatura_c"], color="#e94560", linewidth=0.8)
axes[0].set_ylabel("Temperatura (°C)")
axes[0].axhline(df["temperatura_c"].mean(), color="gray", linestyle="--", linewidth=0.8, label=f'Media: {df["temperatura_c"].mean():.2f}°C')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].plot(df["tiempo"], df["humedad_pct"], color="#0f3460", linewidth=0.8)
axes[1].set_ylabel("Humedad relativa (%)")
axes[1].set_xlabel("Marca de tiempo")
axes[1].axhline(df["humedad_pct"].mean(), color="gray", linestyle="--", linewidth=0.8, label=f'Media: {df["humedad_pct"].mean():.2f}%')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("figuras_informe/series_temporales_act3.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Histogramas de distribución
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Distribución de variables capturadas", fontsize=13, fontweight="bold")

axes[0].hist(df["temperatura_c"], bins=30, color="#e94560", edgecolor="white", alpha=0.85)
axes[0].set_xlabel("Temperatura (°C)")
axes[0].set_ylabel("Frecuencia")
axes[0].set_title("Distribución de temperatura")
axes[0].grid(True, alpha=0.3)

axes[1].hist(df["humedad_pct"], bins=30, color="#0f3460", edgecolor="white", alpha=0.85)
axes[1].set_xlabel("Humedad relativa (%)")
axes[1].set_ylabel("Frecuencia")
axes[1].set_title("Distribución de humedad")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("figuras_informe/histogramas_act3.png", dpi=150, bbox_inches="tight")
plt.show()

---

## 7. Análisis de resultados

Los datos capturados muestran una distribución uniforme dentro de los rangos configurados en CounterFit (temperatura: 18–35 °C, humedad: 40–90 %), lo cual es consistente con el modo *Random* del simulador.

Cada lectura siguió el siguiente flujo:
1. **CounterFit** generó el valor aleatorio del sensor DHT11 virtual.
2. **`nodo_mqtt.py`** leyó ese valor mediante el shim y generó una marca de tiempo en formato ISO 8601.
3. El valor se **persistió en SQLite** (`demo.db`) con `commit()` inmediato, garantizando que ninguna falla de red pudiera causar pérdida de datos.
4. El valor se **publicó vía MQTT** al broker `io.adafruit.com`, donde el feed correspondiente lo recibió y actualizó el dashboard en tiempo real.

La consola mostró confirmación `-> MQTT OK` en cada lectura, indicando que la transmisión fue exitosa en el 100% de los casos durante la prueba.

---

## 8. Conclusiones

- Se implementó exitosamente la cadena completa de un sistema IoT: **captura → almacenamiento local → transmisión a la nube → visualización en tiempo real**.

- El protocolo **MQTT** demostró ser eficiente para telemetría continua gracias a su conexión TCP persistente, bajo overhead y modelo pub/sub desacoplado.

- El enfoque de hardware virtual (**CounterFit + PC**) replica fielmente el comportamiento de un microcontrolador físico **ESP32** con sensor DHT11, permitiendo implementar y validar el sistema completo sin necesidad de componentes físicos.

- La doble persistencia (**SQLite + Adafruit IO**) garantiza que los datos estén disponibles tanto localmente como en la nube, independientemente de interrupciones de red.

- Como mejora futura se propone migrar a hardware físico (ESP32 + DHT11 real), usar MQTT con TLS (puerto 8883) para cifrar la transmisión, e implementar alertas automáticas por umbrales en Adafruit IO.

---

## 9. Referencias

- CounterFit-IoT. (s.f.). *CounterFit* [Repositorio]. GitHub. https://github.com/CounterFit-IoT/CounterFit
- Adafruit Industries. (s.f.). *Adafruit IO* [Plataforma IoT]. https://io.adafruit.com
- OASIS. (2019). *MQTT Version 5.0* [Estándar]. https://docs.oasis-open.org/mqtt/mqtt/v5.0/mqtt-v5.0.html
- Microsoft. (s.f.). *IoT for Beginners* [Repositorio]. GitHub. https://github.com/microsoft/IoT-For-Beginners
- Python Software Foundation. (s.f.). *sqlite3 — DB-API 2.0 interface for SQLite databases*. https://docs.python.org/3/library/sqlite3.html